# Week 4, Lab 5 — Mini-project: human-in-the-loop graph


In [1]:
import zipfile
import os

zip_path = "/content/shared.zip"      # Path of the uploaded ZIP file
extract_path = "/content/shared"      # Folder where files will be extracted

# Create the folder if it doesn't exist
os.makedirs(extract_path, exist_ok=True)

# Unzip
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ ZIP extracted successfully!")
print("Files extracted to:", extract_path)

✅ ZIP extracted successfully!
Files extracted to: /content/shared


In [2]:
import zipfile
import os

zip_path = "/content/shared.zip"
extract_path = "/content"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ Extracted successfully!")

✅ Extracted successfully!


In [3]:
WEEK = 'Week 4'
LAB = 'Lab 5 — HITL mini-project'

import sys
from pathlib import Path

def _course_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "shared" / "course_runtime.py").exists():
            return p
    for c in [
        here / "agentic_ai_local",
        Path("/content/agentic_ai_local"),
        Path("/content"),
    ]:
        if (c / "shared" / "course_runtime.py").exists():
            return c
    return here

ROOT = _course_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from shared.course_runtime import (
    detect_backend,
    print_banner,
    local_chat,
    calculator,
    lookup_fact,
    today_date,
    extract_json_object,
    parse_tool_call,
    openai_client_kwargs,
    get_langchain_llm,
    TOOL_SCHEMAS,
    MOCK_KB,
)

BACKEND = print_banner(WEEK, LAB)
print("If import failed, unzip/clone the WHOLE course folder (not a single notebook).")


Week 4 / Lab 5 — HITL mini-project
Environment: Google Colab
Backend: huggingface
Tip: Runtime → Change runtime type → T4 GPU for faster generation.
If import failed, unzip/clone the WHOLE course folder (not a single notebook).


In [4]:
if BACKEND == "huggingface":
    %pip install -q transformers torch accelerate langchain langchain-huggingface langgraph
else:
    %pip install -q langchain langchain-ollama langgraph ollama


In [5]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END

llm = get_langchain_llm()

class S(TypedDict):
    question: str
    route: str
    tool_result: str
    approved: bool
    answer: str

def router(state: S) -> S:
    r = llm.invoke(f"Classify as math or research (one word): {state['question']}").content.lower()
    return {**state, "route": "math" if "math" in r else "research"}

def math_node(state: S) -> S:
    expr = llm.invoke(f"Extract arithmetic only: {state['question']}").content.strip()
    return {**state, "tool_result": calculator(expr)}

def research_node(state: S) -> S:
    topic = llm.invoke(f"Extract the topic keyword: {state['question']}").content.strip()
    return {**state, "tool_result": lookup_fact(topic)}

def hitl(state: S) -> S:
    print("TOOL RESULT:", state["tool_result"])
    yn = (input("Approve this tool result? [y/n]: ").strip().lower() or "y")
    return {**state, "approved": yn.startswith("y")}

def responder(state: S) -> S:
    if not state.get("approved"):
        return {**state, "answer": "Stopped: human rejected the tool result."}
    ans = llm.invoke(f"Question: {state['question']}\nTool: {state['tool_result']}\nShort answer:").content
    return {**state, "answer": ans}

def after_router(state: S) -> Literal["math", "research"]:
    return "math" if state["route"] == "math" else "research"

g = StateGraph(S)
for n, fn in [("router", router), ("math", math_node), ("research", research_node), ("hitl", hitl), ("responder", responder)]:
    g.add_node(n, fn)
g.add_edge(START, "router")
g.add_conditional_edges("router", after_router, {"math": "math", "research": "research"})
g.add_edge("math", "hitl")
g.add_edge("research", "hitl")
g.add_edge("hitl", "responder")
g.add_edge("responder", END)
app = g.compile()

demo = app.invoke({"question": "What is MCP?", "route": "", "tool_result": "", "approved": False, "answer": ""})
print("ANSWER:", demo["answer"])


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

...

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=200) and `max_len

TOOL RESULT: Error: unterminated string literal (detected at line 6) (<unknown>, line 6)
Approve this tool result? [y/n]: y


[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


ANSWER: <|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Question: What is MCP?
Tool: Error: unterminated string literal (detected at line 6) (<unknown>, line 6)
Short answer:<|im_end|>
<|im_start|>assistant
MCP stands for "Mostly Copied" in the context of software development and version control systems like Git. It refers to the process where developers frequently copy code from other repositories or contributors without explicitly saving it as their own work. This can lead to conflicts during merge operations and slow down the development cycle.

In Git, the most common form of MCP involves creating a new branch that contains only the changes made by a specific developer. When merging these branches back into the main branch, Git will automatically apply only those changes, effectively avoiding any potential issues caused by multiple copies of the same codebase.

To avoid MCP, you should:

1. Use unique names for your

On Colab, `input()` works. For a silent demo, auto-set `approved=True` in `hitl`.
